# 08. Mini-Capstone: an Automated Processing Workflow
**Engineering Data Processing with Python** | one-day course | approx. 60 minutes

> **SOLUTION NOTEBOOK.** Every blank and TODO is filled in. Try the exercise version first; use this when you are stuck or to compare approaches. There is usually more than one correct answer.


In [1]:
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = ""   # if your trainer gives you a download link for data.zip, paste it between the quotes

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

Data folder found: ['3dprinter.csv', 'asset_register.xlsx', 'boiler.json', 'capstone', 'checkpoints', 'concrete.csv', 'manifest.TXT', 'manifest.csv', 'sensor_readings.csv', 'steel.csv', 'thermal-profiles.xlsx', 'work_orders_cmms.csv', 'work_orders_legacy.xlsx']


## The brief

> **From:** Operations Manager, Kilbrack Engineering
> **Subject:** Galway September numbers
>
> Galway runs a different maintenance system from Cork. Their September 2026 export is attached
> (`data/capstone/wo_galway_2026_09.csv`). I need, by Friday:
>
> 1. A clean dataset I can trust, and a list of anything you had to exclude or that Galway needs to fix.
> 2. Work orders by type, mean time to restore for breakdowns by asset type, and the five assets with the most
>    breakdowns.
> 3. A short note on what you did and anything I should be careful about.
>
> I'll want the same every month, so please don't do it by hand.

You have everything you need from Modules 01 to 07. This notebook gives you **less scaffolding** on purpose:
the steps are listed, and there are hints and Gemini prompts, but you write most of the code.

### What "done" looks like

| # | Deliverable | Check |
|---|---|---|
| 1 | `outputs/galway_2026_09/` created by **running the pipeline with a config**, not by hand-editing data | Folder has clean data, KPI-ready data, issues, report, log |
| 2 | `config_galway.json` | Only the config differs from the Cork run, not `pipeline.py` |
| 3 | `kpis.xlsx` in the run folder | Three sheets: by type, time to restore by asset type, top 5 assets |
| 4 | Data note (markdown cell at the end) | Says what was excluded, why, and what you assumed |

Suggested timing: 10 min looking at the data, 15 min config and runs, 20 min KPIs, 10 min data note, 5 min spare.

## 0. Get the pipeline into this session

Each Colab notebook runs on its own machine, so the `pipeline.py` you wrote in Module 07 is not here.
You have two options:

* **Use your own:** download `pipeline.py` from the Module 07 session (Files panel, right-click, Download) and
  upload it here (Files panel, upload icon), then run the next cell.
* **Use the reference version:** just run the next cell. If there is no `pipeline.py` here, it copies the trainer's.

In [2]:
import shutil, json, sys, importlib
from pathlib import Path

for name in ["pipeline.py", "config_cmms.json"]:
    if not Path(name).exists():
        shutil.copy(f"data/checkpoints/{name}", name)
        print(f"Using reference {name}")
    else:
        print(f"Using your own {name}")

import pipeline
importlib.reload(pipeline)
with open("config_cmms.json", encoding="utf-8") as f:
    cmms_config = json.load(f)
print("Pipeline and Cork config loaded.")

Using reference pipeline.py
Using reference config_cmms.json
Pipeline and Cork config loaded.


## 1. Look at the raw export first  [Level 2]

Before writing any config, find out what you are dealing with. Answer these in your head (or in a markdown cell):

* What separates the columns?
* What is the decimal separator for hours and cost?
* What are the column names, and which of ours does each one correspond to?
* What date formats appear in `Opened`, `Closed` and `Modified`?
* What work order type words are used?

Hint: open the file as plain text first (`open(...).readlines()[:4]`). It shows you the raw characters before
pandas interprets anything.

In [3]:
RAW = "data/capstone/wo_galway_2026_09.csv"

with open(RAW, encoding="utf-8") as f:
    for line in f.readlines()[:4]:
        print(line.rstrip())

galway_raw = pd.read_csv(RAW, sep=";", dtype=str)
print(galway_raw.shape)
display(galway_raw.head())
print(galway_raw["Category"].value_counts())
print(galway_raw["Opened"].sample(8, random_state=0).tolist())
print(galway_raw["Closed"].dropna().head(3).tolist(), galway_raw["Modified"].head(3).tolist())

WorkOrder;AssetTag;Category;Prio;Opened;Closed;Hours;Cost EUR;Tech;Details;State;Modified
GWY-0027;CMP-003;Breakdown;3;06/09/2026 06:45;2026-09-06T21:56:39;3,5;637,66;Declan Murphy;Oil leak at separator;Closed;2026-09-06T23:56:39
GWY-0124;CNC-001;Planned Maint.;4;2026-09-25T12:00:00;;6;211,08;D. Murphy;Grease bearings, check alignment;Open;2026-09-25T14:00:00
GWY-0075;HYP-008;Planned Maint.;3;2026-09-15T08:45:00;2026-09-17T12:39:03;4,5;181,78;Murphy, Declan;Grease bearings, check alignment;Closed;2026-09-17T14:39:03
(164, 12)


,WorkOrder,AssetTag,Category,Prio,Opened,Closed,Hours,Cost EUR,Tech,Details,State,Modified
0,GWY-0027,CMP-003,Breakdown,3,06/09/2026 06:45,2026-09-06T21:56:39,"3,5","637,66",Declan Murphy,Oil leak at separator,Closed,2026-09-06T23:56:39
1,GWY-0124,CNC-001,Planned Maint.,4,2026-09-25T12:00:00,NaN,6,"211,08",D. Murphy,"Grease bearings, check alignment",Open,2026-09-25T14:00:00
2,GWY-0075,HYP-008,Planned Maint.,3,2026-09-15T08:45:00,2026-09-17T12:39:03,"4,5","181,78","Murphy, Declan","Grease bearings, check alignment",Closed,2026-09-17T14:39:03
3,GWY-0131,CNC-007,Planned Maint.,4,2026-09-27T06:00:00,NaN,"4,5","77,10",Grainne Fitzgerald,Annual service per OEM schedule,Open,2026-09-27T08:00:00
4,GWY-0097,PMP-002,Planned Maint.,4,2026-09-19T13:30:00,2026-09-19T20:39:03,"3,5","0,48",Aoife Byrne,Annual service per OEM schedule,Closed,2026-09-19T22:39:03


Category
Planned Maint.    38
PPM               30
PM                26
Reactive          20
CM                17
Breakdown         15
INSP               9
Inspection         9
Name: count, dtype: int64
['2026-09-29T09:45:00', '2026-09-05T13:30:00', '2026-09-07T13:30:00', '2026-09-25T07:30:00', '2026-09-21T07:50:00', '03/09/2026 17:50', '2026-09-27T13:20:00', '2026-09-28T14:50:00']
['2026-09-06T21:56:39', '2026-09-17T12:39:03', '2026-09-19T20:39:03'] ['2026-09-06T23:56:39', '2026-09-25T14:00:00', '2026-09-17T14:39:03']


*What you should have found:* semicolon separators, **decimal commas** (`3,5` hours, `1.250,00` euro), different
column names, `Opened` mostly ISO with a `T` (`2026-09-25T12:00:00`) but some `dd/mm/yyyy HH:MM`, and new type
words (`PPM`, `Planned Maint.`, `Reactive`).

## 2. Write the Galway config  [Level 2]

Start from a copy of the Cork config and change only what is different. Everything you found in step 1 goes here.
Leave `type_map` alone for now. You will see why in step 3.

Format codes reminder: `%Y-%m-%dT%H:%M:%S` matches `2026-09-25T12:00:00`.

In [4]:
galway = json.loads(json.dumps(cmms_config))
galway["run_name"] = "galway_2026_09"

galway["source"]["path"] = "data/capstone/wo_galway_2026_09.csv"
galway["source"]["sep"] = ";"
galway["source"]["decimal"] = ","
galway["source"]["date_formats"] = ["%Y-%m-%dT%H:%M:%S", "%d/%m/%Y %H:%M"]
galway["source"]["modified_formats"] = ["%Y-%m-%dT%H:%M:%S"]
galway["source"]["column_map"] = {
    "WorkOrder": "wo_id", "AssetTag": "asset_id", "Category": "wo_type", "Prio": "priority",
    "Opened": "date_raised", "Closed": "date_closed", "Hours": "labour_hours", "Cost EUR": "parts_cost_eur",
    "Tech": "technician", "Details": "description", "State": "status", "Modified": "last_modified",
}

with open("config_galway.json", "w", encoding="utf-8") as f:
    json.dump(galway, f, indent=2, ensure_ascii=False)
print(json.dumps(galway["source"], indent=2))

{
  "path": "data/capstone/wo_galway_2026_09.csv",
  "sep": ";",
  "decimal": ",",
  "date_formats": [
    "%Y-%m-%dT%H:%M:%S",
    "%d/%m/%Y %H:%M"
  ],
  "modified_formats": [
    "%Y-%m-%dT%H:%M:%S"
  ],
  "column_map": {
    "WorkOrder": "wo_id",
    "AssetTag": "asset_id",
    "Category": "wo_type",
    "Prio": "priority",
    "Opened": "date_raised",
    "Closed": "date_closed",
    "Hours": "labour_hours",
    "Cost EUR": "parts_cost_eur",
    "Tech": "technician",
    "Details": "description",
    "State": "status",
    "Modified": "last_modified"
  }
}


## 3. Run it, and let it fail properly

In [5]:
!{sys.executable} pipeline.py config_galway.json

2026-10-06 13:42:55,831 INFO    Run 'galway_2026_09' started
2026-10-06 13:42:55,834 INFO    Loaded 164 rows from data/capstone/wo_galway_2026_09.csv
Traceback (most recent call last):
  File "/sessions/quirky-hopeful-bohr/mnt/outputs/build/_run/pipeline.py", line 211, in <module>
    run_pipeline(config)
  File "/sessions/quirky-hopeful-bohr/mnt/outputs/build/_run/pipeline.py", line 196, in run_pipeline
    df = clean_work_orders(df, cfg)
  File "/sessions/quirky-hopeful-bohr/mnt/outputs/build/_run/pipeline.py", line 85, in clean_work_orders
    raise ValueError(f"Unmapped work order types {unmapped}. Add them to type_map in the config.")
ValueError: Unmapped work order types ['PLANNED MAINT.', 'PPM', 'REACTIVE']. Add them to type_map in the config.


It should stop with **Unmapped work order types**, listing the new words. That is the pipeline doing its job:
it refuses to guess what `Reactive` means. You decide.

### [Level 1: fill the blanks]  Add the new words to the Galway type map, save, run again

In [6]:
galway["type_map"]["PPM"] = "PM"
galway["type_map"]["PLANNED MAINT."] = "PM"
galway["type_map"]["REACTIVE"] = "CM"
with open("config_galway.json", "w", encoding="utf-8") as f:
    json.dump(galway, f, indent=2, ensure_ascii=False)

!{sys.executable} pipeline.py config_galway.json

2026-10-06 13:42:56,603 INFO    Run 'galway_2026_09' started
2026-10-06 13:42:56,605 INFO    Loaded 164 rows from data/capstone/wo_galway_2026_09.csv
2026-10-06 13:42:56,616 INFO    Dedup: 6 exact duplicates, 8 older versions removed, 150 work orders


2026-10-06 13:42:56,785 WARNING Rule closed_before_raised: 1 rows
2026-10-06 13:42:56,785 WARNING Rule negative_hours: 1 rows
2026-10-06 13:42:56,786 WARNING Rule implausible_hours: 1 rows
2026-10-06 13:42:56,786 WARNING Rule unknown_asset: 2 rows
2026-10-06 13:42:56,786 WARNING Rule implausible_cost: 1 rows
2026-10-06 13:42:56,786 INFO    Rule missing_hours: 1 rows
2026-10-06 13:42:56,798 INFO    Outputs written to outputs/galway_2026_09
2026-10-06 13:42:56,798 INFO    Finished: {'rows_out': 150, 'errors': 6, 'output_dir': 'outputs/galway_2026_09'}


In [7]:
run_dir = Path("outputs/galway_2026_09")
files = {p.name for p in run_dir.iterdir()} if run_dir.exists() else set()
report = pd.read_csv(run_dir / "dq_report.csv") if (run_dir / "dq_report.csv").exists() else None
check({"work_orders_clean.csv", "work_orders_kpi_ready.csv", "dq_issues.csv", "dq_report.csv", "pipeline.log"} <= files,
      "Galway processed by config alone. pipeline.py did not change.",
      "Outputs missing. Read the error from the run above: it names the setting to fix.")
report

PASS: Galway processed by config alone. pipeline.py did not change.


,rule,severity,rows_flagged,pct
0,closed_before_raised,error,1,0.67
1,negative_hours,error,1,0.67
2,implausible_hours,error,1,0.67
3,closed_without_date,error,0,0.00
4,unknown_asset,error,2,1.33
5,implausible_cost,error,1,0.67
6,negative_cost,warning,0,0.00
7,missing_hours,warning,1,0.67
8,missing_cost,warning,0,0.00
9,open_too_long,warning,0,0.00


## 4. Read your own quality report  [Level 2]

Open `dq_issues.csv` and look at the error rows. For each error rule that fired, decide: is it a data entry
problem Galway should fix, or something you need to ask about? (For example: is 300 hours a typo for 3.00, or
300 minutes? You cannot know. That is why it goes back to Galway rather than being "fixed" by you.)

In [8]:
issues = pd.read_csv(run_dir / "dq_issues.csv")
issues[issues["dq_error"]][["wo_id", "asset_id", "wo_type", "date_raised", "date_closed",
                            "labour_hours", "parts_cost_eur", "dq_issues"]]

,wo_id,asset_id,wo_type,date_raised,date_closed,labour_hours,parts_cost_eur,dq_issues
0,GWY-0006,CMP-007,CM,2026-09-01 16:20:00,2026-09-02 12:25:35,300.0,1081.58,implausible_hours
1,GWY-0018,CMP-003,CM,2026-09-03 17:50:00,2026-09-04 16:18:43,-3.0,67.06,negative_hours
2,GWY-0023,CNC-013,PM,2026-09-04 13:00:00,2026-09-06 16:56:17,3.5,99999.00,implausible_cost
3,GWY-0031,CHL-099,PM,2026-09-06 11:30:00,2026-09-08 21:28:34,4.0,17.25,unknown_asset
4,GWY-0042,PMP-099,PM,2026-09-08 16:00:00,2026-09-10 16:36:31,5.5,278.77,unknown_asset
5,GWY-0049,CNC-014,PM,2026-09-10 12:50:00,2026-08-30 10:00:00,4.0,228.25,closed_before_raised


## 5. KPIs  [Level 3: write it yourself, Gemini allowed]

Using **only KPI-ready rows** (`work_orders_kpi_ready.csv`):

* **(a) Work orders by type**: count of PM, CM, INSP.
* **(b) Mean time to restore for breakdowns by asset type**: for **closed** CM work orders,
  hours from `date_raised` to `date_closed`. Show count, mean and median per `asset_type`.
* **(c) Top 5 assets by number of CM work orders.**

Save all three to `kpis.xlsx` in the run folder, one sheet each (`pd.ExcelWriter`).

**Prompt to try** (paste `ready.dtypes` and `ready.head()` first so Gemini knows the columns):
> Using this pandas DataFrame `ready`, calculate (a) counts by wo_type, (b) for closed CM rows, hours between
> date_raised and date_closed, with count, mean and median by asset_type, (c) the top 5 asset_id by number of CM
> rows. Save the three tables as sheets in one Excel file. Explain each step and tell me what you assumed.

**Check the AI's answer:**
* Did it filter to CM **and** closed before calculating time to restore?
* Did it parse the date columns as dates when loading the CSV? (If not, the subtraction fails or is wrong.)
* Do the counts in (a) add up to the number of KPI-ready rows?
* Why is median worth showing alongside mean for repair times?

In [9]:
ready = pd.read_csv(run_dir / "work_orders_kpi_ready.csv", parse_dates=["date_raised", "date_closed"])

by_type = ready["wo_type"].value_counts().rename_axis("wo_type").reset_index(name="work_orders")

cm_closed = ready[(ready["wo_type"] == "CM") & ready["date_closed"].notna()].copy()
cm_closed["hours_to_restore"] = (cm_closed["date_closed"] - cm_closed["date_raised"]).dt.total_seconds() / 3600
restore = (cm_closed.groupby("asset_type")["hours_to_restore"]
                    .agg(["count", "mean", "median"]).round(1)
                    .sort_values("mean", ascending=False).reset_index())

top5 = (ready[ready["wo_type"] == "CM"]["asset_id"].value_counts().head(5)
             .rename_axis("asset_id").reset_index(name="cm_work_orders"))

with pd.ExcelWriter(run_dir / "kpis.xlsx") as xl:
    by_type.to_excel(xl, sheet_name="by_type", index=False)
    restore.to_excel(xl, sheet_name="time_to_restore", index=False)
    top5.to_excel(xl, sheet_name="top5_cm_assets", index=False)

print("KPI-ready rows:", len(ready), " counted by type:", by_type["work_orders"].sum())
display(by_type, restore, top5)

KPI-ready rows: 144  counted by type: 144


,wo_type,work_orders
0,PM,79
1,CM,48
2,INSP,17


,asset_type,count,mean,median
0,Air Compressor,8,32.4,31.9
1,Conveyor,3,27.7,17.4
2,Pump,6,22.8,21.7
3,CNC Mill,20,21.9,20.8
4,Hydraulic Press,2,11.1,11.1


,asset_id,cm_work_orders
0,CNC-012,5
1,CNC-007,5
2,CMP-003,5
3,PMP-004,4
4,PMP-002,4


In [10]:
xl_path = run_dir / "kpis.xlsx"
sheets = pd.read_excel(xl_path, sheet_name=None) if xl_path.exists() else {}
check(len(sheets) == 3 and sheets[list(sheets)[0]].iloc[:, -1].sum() == len(ready),
      "kpis.xlsx has three sheets and the type counts add up to the KPI-ready rows.",
      "Expected three sheets, with counts by type adding up to len(ready).")

PASS: kpis.xlsx has three sheets and the type counts add up to the KPI-ready rows.


## 6. The data note  [Level 3]

Write the note for the operations manager in the markdown cell below. Plain English, five to ten lines. Cover:

* What you did, in one or two sentences (not the code).
* How many records came in, how many are in the KPIs, how many were excluded and why.
* What Galway needs to fix.
* What you **assumed**, and what you **could not determine**. For example: "time to restore" here is raised to
  closed, which includes waiting for parts and for a technician. It is not hands-on repair time.

Then ask Gemini to critique your note: *"You are an operations manager. What in this note is unclear, and what
would you want to know that it does not say?"* Edit, but keep it yours.

*Example data note:*

**Galway work orders, September 2026: data note**

The September export (164 rows) was processed with the same automated pipeline as Cork, using a Galway settings
file. 14 rows were duplicates (6 exact copies, 8 older "Open" versions of jobs that were later closed), leaving 150
work orders. 6 have errors and are **excluded from the KPIs**: two are on asset tags not in the register
(CHL-099, PMP-099), GWY-0006 has 300 labour hours, GWY-0018 has -3 hours, GWY-0023 has a parts cost of €99,999
(looks like a placeholder) and GWY-0049 was closed on 30 August, before it was opened. These are listed in
`dq_issues.csv` for Galway to
correct. One job has no hours recorded; it is kept but flagged.

Assumptions: Galway's "Reactive" means corrective (CM), and "PPM" / "Planned Maint." mean preventive (PM).
"Time to restore" is raised to closed, so it includes waiting time, not just repair time. Open jobs are not in the
time-to-restore figures, so late-month breakdowns that are still open are not counted yet.

## Stretch options

1. **Make KPIs part of the pipeline.** Move your KPI code into a `make_kpis(df, out_dir)` function in
   `pipeline.py`, called at the end of `run_pipeline`. Rerun both configs. Did anything break for Cork?
2. **Next month's export.** Ask Gemini to generate a small fake October file in Galway's format with one new
   problem it does not tell you about. Run the pipeline on it. Did the pipeline catch the problem, or did it get
   through? What rule would catch it?
3. **Sensor link.** Galway has CMP-003, PMP-002 and PMP-004 on condition monitoring. Using Module 06's outputs,
   which would you watch most closely and why?

## What you built today

You went from loading files without errors to a workflow that:

* loads different export formats through **configuration**, not code changes,
* standardises text, categories, numbers, units and dates, and **counts what it could not convert**,
* removes duplicates by a rule you can explain, and sends conflicts to a person,
* combines sources with audit columns and safe joins,
* checks quality with explicit rules, separates errors from warnings, and stops on structural problems,
* writes clean outputs, an issues list, a report and a log, every run, the same way.

And you used AI as a fast assistant whose work you checked, rather than as an oracle.